# Imputation Model Training

Trains and serialises ML models to impute missing environmental variables in incoming farm profiles.

## Sequential Imputation Pipeline

Models are trained and applied in order so that each model only uses features
that are **guaranteed to be available** — either from farm geometry or already imputed in a prior step.

| Step | Target | Type | Model | Features |
|---|---|---|---|---|
| 1 | `elevation_m` | Numeric | KNN Regressor (tuned) | base only |
| 2 | `slope` | Numeric | KNN Regressor | base + elevation_m |
| 3 | `temperature_celsius` | Numeric | KNN Regressor | base + elevation_m |
| 4 | `rainfall_mm` | Numeric | KNN Regressor (tuned) | base + elevation_m + temperature_celsius |
| 5 | `soil_texture` | Categorical | KNN Classifier (tuned) | base + elevation_m + temperature_celsius + rainfall_mm |
| 6 | `ph` | Numeric | Random Forest Regressor | base + all above |

## Base Features (always available from farm geometry)
- `latitude`, `longitude` — derived from farm boundary geometry
- `area_ha` — derived from farm boundary geometry
- `coastal`, `riparian` — derived from GIS overlays

Models are trained using only ground-truth rows (`is_predicted == False`).
Serialised with `joblib` to `src/models/imputation/`.

## 1. Imports

In [21]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
DATA_PATH = Path("../data/farms_cleaned.csv")
MODELS_PATH = Path("../src/models/imputation")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [22]:
df = pd.read_csv(DATA_PATH)
print(f"Total rows: {len(df)}")
print(f"is_predicted breakdown:\n{df['is_predicted'].value_counts()}")

# Use only ground-truth rows for training
df_train = df[df["is_predicted"] == False].copy()
print(f"\nGround-truth rows available for training: {len(df_train)}")
df_train.head()

Total rows: 992
is_predicted breakdown:
is_predicted
False    990
True       2
Name: count, dtype: int64

Ground-truth rows available for training: 990


,id,rainfall_mm,temperature_celsius,elevation_m,ph,soil_texture,area_ha,slope,latitude,longitude,coastal,riparian,is_predicted
0,1,1958,23,585,6.2,clay,0.37,13.114323,-8.568798,126.675704,False,False,False
1,2,1958,23,481,6.2,clay,0.49,13.937272,-8.567578,126.680149,False,False,False
2,3,2020,25,179,8.2,sandy loam,1.22,11.151526,-8.642259,126.664651,False,False,False
3,4,2553,24,259,5.9,sandy loam,0.47,15.552424,-8.639690,126.651458,False,False,False
4,5,2020,25,129,7.0,clay,2.05,7.615144,-8.643001,126.670084,False,True,False


In [23]:
# Confirm no nulls in training set
null_counts = df_train.isnull().sum()
print("Null counts in training set:")
print(null_counts[null_counts > 0] if null_counts.any() else "No nulls — clean training set.")

Null counts in training set:
No nulls — clean training set.


## 3. Encode Categorical Features

`soil_texture` is categorical and needs to be encoded both as a target (for the classifier)
and as a feature (when predicting other targets).
A `LabelEncoder` is saved alongside the model so the service can decode predictions.

In [24]:
le_soil = LabelEncoder()
df_train["soil_texture_enc"] = le_soil.fit_transform(df_train["soil_texture"])

print("Soil texture classes:")
for i, cls in enumerate(le_soil.classes_):
    print(f"  {i}: {cls}")

# Save the encoder so the imputation service can decode predictions
joblib.dump(le_soil, MODELS_PATH / "soil_texture_encoder.joblib")
print("\nSaved: soil_texture_encoder.joblib")

Soil texture classes:
  0: clay
  1: clay loam
  2: loam
  3: sandy clay
  4: sandy loam
  5: silty loam

Saved: soil_texture_encoder.joblib


## 4. Define Feature Sets

**Base features** — always available from farm geometry, used by all models.

**Extended features** — include other environmental columns that may also be available.
The imputation service will use whichever features are non-null in the incoming profile.

In [25]:
BASE_FEATURES = ["latitude", "longitude", "area_ha", "coastal", "riparian"]

# Convert booleans to int for sklearn
df_train["coastal"] = df_train["coastal"].astype(int)
df_train["riparian"] = df_train["riparian"].astype(int)

# Sequential feature sets — each target only uses base features or targets
# imputed in earlier steps, guaranteeing no missing values at inference time.
FEATURE_SETS = {
    "elevation_m":          BASE_FEATURES,
    "slope":                BASE_FEATURES + ["elevation_m"],
    "temperature_celsius":  BASE_FEATURES + ["elevation_m"],
    "rainfall_mm":          BASE_FEATURES + ["elevation_m", "temperature_celsius"],
    "soil_texture":         BASE_FEATURES + ["elevation_m", "temperature_celsius", "rainfall_mm"],
    "ph":                   BASE_FEATURES + ["elevation_m", "temperature_celsius", "rainfall_mm", "soil_texture_enc"],
}

# Ordered list the inference service must follow
IMPUTATION_ORDER = ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "soil_texture", "ph"]

print("Sequential feature sets defined:")
for t in IMPUTATION_ORDER:
    print(f"  {t:25s} <- {FEATURE_SETS[t]}")

Sequential feature sets defined:
  elevation_m               <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian']
  slope                     <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian', 'elevation_m']
  temperature_celsius       <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian', 'elevation_m']
  rainfall_mm               <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian', 'elevation_m', 'temperature_celsius']
  soil_texture              <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian', 'elevation_m', 'temperature_celsius', 'rainfall_mm']
  ph                        <- ['latitude', 'longitude', 'area_ha', 'coastal', 'riparian', 'elevation_m', 'temperature_celsius', 'rainfall_mm', 'soil_texture_enc']


## 5. Train & Evaluate Models

80/20 train/holdout split with a fixed seed for reproducibility.
Models that previously failed thresholds (`elevation_m`, `rainfall_mm`, `soil_texture`) are tuned with `GridSearchCV`.

In [26]:
results = {}

KNN_PARAM_GRID = {
    "n_neighbors": [3, 5, 7, 10, 15, 20],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

# Targets requiring hyperparameter tuning (failed thresholds with default KNN)
TUNE_TARGETS = {"elevation_m", "rainfall_mm"}

NUMERIC_TARGETS = {
    "elevation_m":          KNeighborsRegressor(),
    "slope":                KNeighborsRegressor(n_neighbors=5),
    "temperature_celsius":  KNeighborsRegressor(n_neighbors=5),
    "rainfall_mm":          KNeighborsRegressor(),
    "ph":                   RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED),
}

# Train in sequential order — each model only uses features from prior steps
for target in ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "ph"]:
    model = NUMERIC_TARGETS[target]
    features = FEATURE_SETS[target]
    X = df_train[features]
    y = df_train[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )

    if target in TUNE_TARGETS:
        grid = GridSearchCV(
            model, KNN_PARAM_GRID, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
        )
        grid.fit(X_train, y_train)
        model = grid.best_estimator_
        NUMERIC_TARGETS[target] = model
        print(f"{target:25s} | best params: {grid.best_params_}")
    else:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results[target] = {"RMSE": round(rmse, 3), "MAE": round(mae, 3), "R2": round(r2, 3)}

    joblib.dump(model, MODELS_PATH / f"{target}_imputer.joblib")
    print(f"{target:25s} | RMSE: {rmse:8.3f} | MAE: {mae:8.3f} | R²: {r2:.3f} | saved")

elevation_m               | best params: {'metric': 'manhattan', 'n_neighbors': 10, 'weights': 'distance'}
elevation_m               | RMSE:  152.431 | MAE:  105.728 | R²: 0.628 | saved
slope                     | RMSE:    6.598 | MAE:    5.020 | R²: 0.296 | saved
temperature_celsius       | RMSE:    0.770 | MAE:    0.591 | R²: 0.713 | saved
rainfall_mm               | best params: {'metric': 'manhattan', 'n_neighbors': 20, 'weights': 'distance'}
rainfall_mm               | RMSE:  215.920 | MAE:  165.946 | R²: 0.621 | saved
ph                        | RMSE:    0.247 | MAE:    0.123 | R²: 0.915 | saved


In [27]:
# --- Step 5: KNN Classifier — soil_texture (tuned) ---
# Trained after elevation_m, temperature_celsius, rainfall_mm are imputed,
# so all features are guaranteed available at inference time.
target = "soil_texture"
features = FEATURE_SETS[target]
X = df_train[features]
y = df_train["soil_texture_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

clf_param_grid = {
    "n_neighbors": [3, 5, 7, 10, 15, 20],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}

clf_grid = GridSearchCV(
    KNeighborsClassifier(),
    clf_param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)
clf_grid.fit(X_train, y_train)
clf = clf_grid.best_estimator_
print(f"soil_texture              | best params: {clf_grid.best_params_}")

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

results[target] = {"Accuracy": round(acc, 3), "Macro F1": round(f1, 3)}

joblib.dump(clf, MODELS_PATH / "soil_texture_imputer.joblib")
print(f"{target:25s} | Accuracy: {acc:.3f} | Macro F1: {f1:.3f} | saved")
print()
print(classification_report(y_test, y_pred, target_names=le_soil.classes_))

soil_texture              | best params: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'distance'}
soil_texture              | Accuracy: 0.813 | Macro F1: 0.720 | saved

              precision    recall  f1-score   support

        clay       0.87      0.91      0.89        87
   clay loam       0.45      0.36      0.40        14
        loam       0.75      0.75      0.75         4
  sandy clay       0.79      0.87      0.83        30
  sandy loam       0.81      0.77      0.79        61
  silty loam       1.00      0.50      0.67         2

    accuracy                           0.81       198
   macro avg       0.78      0.69      0.72       198
weighted avg       0.81      0.81      0.81       198



## 6. Results Summary

In [28]:
print("=" * 60)
print("MODEL PERFORMANCE SUMMARY (holdout set, 20%)")
print("=" * 60)

for target, metrics in results.items():
    metric_str = " | ".join(f"{k}: {v}" for k, v in metrics.items())
    print(f"{target:25s} | {metric_str}")

print()
print("Acceptance thresholds:")
print("  rainfall_mm       RMSE < 200mm")
print("  temperature_celsius RMSE < 2°C")
print("  elevation_m       RMSE < 100m")
print("  slope             RMSE < 5°")
print("  ph                RMSE < 1.0")
print("  soil_texture      Accuracy > 0.75, Macro F1 > 0.70")

MODEL PERFORMANCE SUMMARY (holdout set, 20%)
elevation_m               | RMSE: 152.431 | MAE: 105.728 | R2: 0.628
slope                     | RMSE: 6.598 | MAE: 5.02 | R2: 0.296
temperature_celsius       | RMSE: 0.77 | MAE: 0.591 | R2: 0.713
rainfall_mm               | RMSE: 215.92 | MAE: 165.946 | R2: 0.621
ph                        | RMSE: 0.247 | MAE: 0.123 | R2: 0.915
soil_texture              | Accuracy: 0.813 | Macro F1: 0.72

Acceptance thresholds:
  rainfall_mm       RMSE < 200mm
  temperature_celsius RMSE < 2°C
  elevation_m       RMSE < 100m
  slope             RMSE < 5°
  ph                RMSE < 1.0
  soil_texture      Accuracy > 0.75, Macro F1 > 0.70


## 7. Validate Thresholds

In [29]:
THRESHOLDS = {
    "rainfall_mm":          {"RMSE": 200},
    "temperature_celsius":  {"RMSE": 2},
    "elevation_m":          {"RMSE": 100},
    "slope":                {"RMSE": 5},
    "ph":                   {"RMSE": 1.0},
    "soil_texture":         {"Accuracy": 0.75, "Macro F1": 0.70},
}

all_passed = True
for target, thresholds in THRESHOLDS.items():
    for metric, threshold in thresholds.items():
        actual = results[target].get(metric)
        if actual is None:
            continue
        if metric in ("RMSE", "MAE"):
            passed = actual <= threshold
        else:
            passed = actual >= threshold
        status = "PASS" if passed else "FAIL"
        if not passed:
            all_passed = False
        print(f"{target:25s} | {metric:10s} | actual={actual} | threshold={threshold} | {status}")

print()
print("All thresholds passed!" if all_passed else "WARNING: Some models did not meet thresholds — review before deploying.")

rainfall_mm               | RMSE       | actual=215.92 | threshold=200 | FAIL
temperature_celsius       | RMSE       | actual=0.77 | threshold=2 | PASS
elevation_m               | RMSE       | actual=152.431 | threshold=100 | FAIL
slope                     | RMSE       | actual=6.598 | threshold=5 | FAIL
ph                        | RMSE       | actual=0.247 | threshold=1.0 | PASS
soil_texture              | Accuracy   | actual=0.813 | threshold=0.75 | PASS
soil_texture              | Macro F1   | actual=0.72 | threshold=0.7 | PASS



## 8. Save Feature Set Metadata

The imputation service needs to know which features each model expects at inference time.

In [ ]:
# import json

# metadata = {
#     "base_features": BASE_FEATURES,
#     "feature_sets": FEATURE_SETS,
#     "imputation_order": IMPUTATION_ORDER,
#     "numeric_targets": ["elevation_m", "slope", "temperature_celsius", "rainfall_mm", "ph"],
#     "categorical_targets": ["soil_texture"],
#     "random_seed": RANDOM_SEED,
# }

# metadata_path = MODELS_PATH / "metadata.json"
# with open(metadata_path, "w") as f:
#     json.dump(metadata, f, indent=2)

# print(f"Saved metadata to {metadata_path}")
# print()
# print("Imputation order:", IMPUTATION_ORDER)
# print()
# print("Models saved to:", MODELS_PATH)
# for f in sorted(MODELS_PATH.glob("*.joblib")):
#     size_kb = f.stat().st_size / 1024
#     print(f"  {f.name:45s} {size_kb:6.1f} KB")

Saved metadata to ../src/models/imputation/metadata.json

Imputation order: ['elevation_m', 'slope', 'temperature_celsius', 'rainfall_mm', 'soil_texture', 'ph']

Models saved to: ../src/models/imputation
  elevation_m_imputer.joblib                      79.2 KB
  ph_imputer.joblib                             1926.1 KB
  rainfall_mm_imputer.joblib                     105.0 KB
  slope_imputer.joblib                            92.1 KB
  soil_texture_encoder.joblib                      0.5 KB
  soil_texture_imputer.joblib                    118.0 KB
  temperature_celsius_imputer.joblib              92.1 KB
